# MOSES Evaluation Testing

This notebook tests the MOSES (Molecular Sets) evaluation toolkit before integrating with the GRASSY model.

MOSES provides metrics for evaluating molecular generation models:
- **Validity**: Fraction of valid molecules
- **Uniqueness**: Fraction of unique molecules
- **Novelty**: Fraction of molecules not in training set
- **Fréchet ChemNet Distance (FCD)**: Distribution similarity
- **Internal Diversity**: Diversity within generated set
- **Filters**: Drug-likeness filters (MCF, PAINS, etc.)

## References
- MOSES paper: https://arxiv.org/abs/1811.12823
- GitHub: https://github.com/molecularsets/moses

## 1. Install MOSES

First, we need to install the MOSES package.

In [ ]:
# Install MOSES from PyPI
!pip install moses-chem

## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem
from moses import get_all_metrics
import torch

## 3. Load Training Data (ZINC12K)

We need the training SMILES to compute novelty metrics.

In [ ]:
# Load ZINC12K dataset to extract SMILES
zinc_data = np.load('../datasets/ZINC12K.npy', allow_pickle=True).item()

# Extract SMILES strings from training set
train_smiles = []
for mol_data in zinc_data['mol_data']:
    if 'smiles' in mol_data:
        train_smiles.append(mol_data['smiles'])
    elif 'SMILES' in mol_data:
        train_smiles.append(mol_data['SMILES'])

print(f"Loaded {len(train_smiles)} training SMILES")
print(f"Example SMILES: {train_smiles[:3]}")

## 4. Generate Test Molecules

For now, we'll create some dummy generated molecules to test MOSES.
Later, you'll replace this with actual generations from your model.

In [ ]:
# Create dummy "generated" molecules for testing
# In practice, you'll generate these from your model

# Option 1: Use a subset of training molecules (will have low novelty)
test_generated = train_smiles[:100]

# Option 2: Add some simple variations for testing
# Uncomment to test with simple molecules
# test_generated = [
#     'CCO',  # Ethanol
#     'CC(C)O',  # Isopropanol
#     'c1ccccc1',  # Benzene
#     'CC(=O)O',  # Acetic acid
#     'c1ccc(O)cc1',  # Phenol
# ] * 20  # Repeat to get 100 molecules

print(f"Generated {len(test_generated)} test molecules")

## 5. Run MOSES Evaluation

Compute all MOSES metrics on the generated molecules.

In [ ]:
# Run all MOSES metrics
print("Computing MOSES metrics...")
print("This may take a few minutes...\n")

metrics = get_all_metrics(
    gen=test_generated,           # Generated SMILES
    train=train_smiles,           # Training SMILES (for novelty)
    test=train_smiles[:1000],     # Test set (for FCD computation)
    k=len(test_generated),        # Number of molecules to consider
    device='cpu',                 # Use 'cuda' if GPU available
    batch_size=512,               # Batch size for FCD computation
    n_jobs=1                      # Number of parallel jobs
)

print("\n" + "="*70)
print("MOSES EVALUATION METRICS")
print("="*70)

## 6. Display Results

In [ ]:
# Convert metrics to DataFrame for better display
metrics_df = pd.DataFrame([metrics]).T
metrics_df.columns = ['Value']

# Group metrics by category
print("\n📊 Basic Metrics:")
print("-" * 50)
basic_metrics = ['valid', 'unique@1000', 'unique@10000', 'Novelty']
for metric in basic_metrics:
    if metric in metrics:
        print(f"  {metric:20s}: {metrics[metric]:.4f}")

print("\n📈 Diversity Metrics:")
print("-" * 50)
diversity_metrics = ['IntDiv', 'IntDiv2']
for metric in diversity_metrics:
    if metric in metrics:
        print(f"  {metric:20s}: {metrics[metric]:.4f}")

print("\n🔬 Distribution Metrics:")
print("-" * 50)
dist_metrics = ['FCD/Test', 'FCD/TestSF']
for metric in dist_metrics:
    if metric in metrics:
        print(f"  {metric:20s}: {metrics[metric]:.4f}")

print("\n💊 Drug-likeness Filters:")
print("-" * 50)
filter_metrics = ['Filters', 'logP', 'SA', 'QED', 'weight', 'MCF', 'PAINS']
for metric in filter_metrics:
    if metric in metrics:
        print(f"  {metric:20s}: {metrics[metric]:.4f}")

print("\n" + "="*70)

# Display full table
print("\nFull Metrics Table:")
display(metrics_df)

## 7. Interpret Results

**Key Metrics Explanation:**

- **valid**: Fraction of chemically valid SMILES (target: ~1.0)
- **unique@k**: Fraction of unique molecules among k samples (target: high)
- **Novelty**: Fraction not in training set (target: 0.7-0.9)
- **FCD**: Fréchet ChemNet Distance (target: <1.0 is excellent)
- **IntDiv**: Internal diversity using Tanimoto similarity (target: high)
- **Filters**: Fraction passing drug-likeness filters (target: high)
- **QED**: Quantitative Estimate of Drug-likeness (target: 0.4-0.6)
- **SA**: Synthetic Accessibility (target: 2.5-3.5)

Lower is better for: **FCD**, **SA** (to some extent)  
Higher is better for: **valid**, **unique**, **Novelty**, **IntDiv**, **Filters**, **QED**

## 8. Next Steps: Integrate Your Model

Once you're satisfied with the MOSES setup, you can:

1. Load your trained GRASSY model
2. Generate molecules by:
   - Sampling from the latent space
   - Decoding latent vectors to molecular representations
   - Converting to SMILES strings
3. Run MOSES evaluation on your generated molecules

```python
# Placeholder for model integration
# from models.GRASSY_model import GRASSY
# 
# model = GRASSY.load_from_checkpoint('path/to/checkpoint.ckpt')
# model.eval()
# 
# with torch.no_grad():
#     # Generate latent samples
#     z = torch.randn(1000, latent_dim)
#     # Decode to molecules
#     generated_mols = model.decode(z)
#     # Convert to SMILES
#     generated_smiles = [mol_to_smiles(mol) for mol in generated_mols]
# 
# # Evaluate
# metrics = get_all_metrics(gen=generated_smiles, train=train_smiles, ...)
```

In [ ]:
# Placeholder cell for model loading and generation
# Add your model code here once MOSES testing is complete
pass